#Predicting sewage levels

In [ ]:
import numpy as np
import pandas as pd

import random
import os
import wandb
import math
import pprint

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import cross_val_score, KFold
from sklearn.feature_selection import SelectKBest, mutual_info_regression

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV, cross_val_score
from sklearn.linear_model import LinearRegression
from scipy.stats import randint
# The below are for classification
# from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report


random_state = 138
random.seed(random_state)

The paths can be configured to run locally.

In [ ]:
Pluv_2022_path = 'Pluviometer data/2022.xlsx'
Pluv_2023_path = 'Pluviometer data/2023.xlsx'
Pluv_2024_path = 'Pluviometer data/2024.xlsx'

U24_2022_path = 'Sewage data/U24_2022.csv'
U24_2023_path = 'Sewage data/U24_2023.csv'
U24_2024_path = 'Sewage data/U24_2024.csv'


In [ ]:
import os
print(os.getcwd())  # Zeigt den aktuellen Pfad
print(os.listdir('.'))  # Listet alle Dateien im aktuellen Ordner auf

##**Rainfall data from Flagey and Avant Port**

Removing empty columns.

In [ ]:
pd_2022_df = pd.read_excel(Pluv_2022_path)
pd_2023_df = pd.read_excel(Pluv_2023_path)
pd_2024_df = pd.read_excel(Pluv_2024_path)

pd_2022_df.drop(['Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8'], axis=1, inplace=True)
pd_2022_df = pd_2022_df.rename(columns={'U24': 'P01', 'C12': 'P14'})

In [ ]:
pd_merged_df = pd.concat([pd_2022_df, pd_2023_df, pd_2024_df], ignore_index=True)
pd_merged_df.columns.values

In [ ]:
pd_merged_df = pd_merged_df.rename(columns={"FLOWBRU - PLUVIO ALL":"datetime", "P01":"avant_port_mm", "P14":"flagey_mm"})

###**Data cleaning**


Initially, it was assumed that the datetime inconsistency could be
corrected through interpolation, but this assumption turned out to be incorrect.

In [ ]:
for i in range(315650, 315655):
    print(f"Row {i}: {pd_merged_df.iloc[i]['datetime']}")
    print("-" * 40)

One of the datasets has a invalid first row.

In [ ]:
print(f"Row {0}: {pd_merged_df.iloc[0]['datetime']}")

Interpolation is not appropriate here: it cannot realistically reconstruct
5-minute interval data from the totals.

Attempted datetime correction (invalid approach for this dataset)
```
pd_merged_df['datetime'] = pd.to_datetime(pd_merged_df['datetime'], errors='coerce')

for i in range(1, len(pd_merged_df)):
    if pd.isna(pd_merged_df.loc[i, 'datetime']):
        pd_merged_df.loc[i, 'datetime'] = (
            pd_merged_df.loc[i - 1, 'datetime'] + pd.Timedelta(minutes=5)
        )
```

Since the data is unreliable, we remove the affected rows instead.
Note: Removing rows by index in a loop can lead to shifting index issues, so we drop them all at once.

*errors='coerce'* makes invalid dates turn into NaN, so you can easily find and remove them later.

In [ ]:
pd_merged_df['datetime'] = pd.to_datetime(pd_merged_df['datetime'], errors='coerce')
pd_merged_df['avant_port_mm'] = pd.to_numeric(pd_merged_df['avant_port_mm'], errors='coerce')
pd_merged_df['flagey_mm'] = pd.to_numeric(pd_merged_df['flagey_mm'], errors='coerce')

pre_count = pd_merged_df['datetime'].isna().sum()
pd_merged_df = pd_merged_df.dropna()
print(f"Deleted rows count: {pre_count}")

In [ ]:
pd_merged_df['datetime']= pd.to_datetime(pd_merged_df['datetime'])
pd_merged_df['avant_port_mm'] = pd.to_numeric(pd_merged_df['avant_port_mm'])
pd_merged_df['flagey_mm'] = pd.to_numeric(pd_merged_df['flagey_mm'])

In [ ]:
pd_merged_df = pd_merged_df.sort_values("datetime").reset_index(drop=True)
pd_merged_df = pd_merged_df.set_index('datetime')
pd_merged_df.head()

###**Exploratory Data Analaysis**

In [ ]:
print(pd_merged_df[["avant_port_mm", "flagey_mm"]].describe())

corr = pd_merged_df["avant_port_mm"].corr(pd_merged_df["flagey_mm"])
print(f"Correlation: {corr:.3f}")

diff = (pd_merged_df["avant_port_mm"] - pd_merged_df["flagey_mm"]).abs()
print(f"Max difference: {diff.max():.4f} mm")

print(f"Null count: {pd_merged_df.isnull().sum()}")
print(f"Duplicate count: {pd_merged_df.duplicated().sum()}")

Checking for unrealistic rain volume in 5-minutes intervals.

In [ ]:
limit = 10
bef_count = pd_merged_df.count()

old_pd_merged_df = pd_merged_df

pd_merged_df = pd_merged_df[
    (pd_merged_df['flagey_mm'] <= limit) |
    (pd_merged_df['avant_port_mm'] <= limit)
]
print(f"Deleted rows: \n{bef_count - pd_merged_df.count()}")

Checking for unrealistic 5-minutes interval differences.

In [ ]:
pd_merged_df['delta_avant_port_mm'] = pd_merged_df['avant_port_mm'].diff()
pd_merged_df['delta_flagey_mm'] = pd_merged_df['flagey_mm'].diff()

In [ ]:
limit = 6
bef_count = pd_merged_df.count()

pd_merged_df = pd_merged_df[
    (pd_merged_df['delta_flagey_mm'] <= limit) |
    (pd_merged_df['delta_avant_port_mm'] <= limit)
]
print(f"Deleted rows: \n{bef_count - pd_merged_df.count()}")

Data distribution after removing outliers.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12,6))

sns.boxplot(old_pd_merged_df['flagey_mm'], ax=axes[0,0])
axes[0,0].set_title('Flagey Rainfall (before removing outliners)')

sns.boxplot(old_pd_merged_df['avant_port_mm'], ax=axes[0,1])
axes[0,1].set_title('Avant Port Rainfall (before removing outliners)')

sns.boxplot(pd_merged_df['flagey_mm'], ax=axes[1,0])
axes[1,0].set_title('Flagey Rainfall (after removing outliners)')

sns.boxplot(pd_merged_df['avant_port_mm'], ax=axes[1,1])
axes[1,1].set_title('Avant Port Rainfall (after removing outliners)')

plt.show()

We do this because we need to use the datetime values for accurate plotting, and resetting the index makes it easier to access and work with the datetime column.

In [ ]:
pd_merged_df = pd_merged_df.reset_index(drop=False)
pd_merged_df.head(1)

We are plotting rainfall volume from two sensors. We use separate charts because the data from both sensors is very similar, and combining them would make it harder to distinguish differences.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

ax1.plot(pd_merged_df["datetime"], pd_merged_df["avant_port_mm"], "steelblue", linewidth=0.8)
ax1.set_ylabel("Rainfall (mm)")
ax1.set_title("Avant Port")

ax2.plot(pd_merged_df["datetime"], pd_merged_df["flagey_mm"], "crimson", linewidth=0.8)
ax2.set_ylabel("Rainfall (mm)")
ax2.set_title("Flagey")

plt.show()

###**Feature Engineering / Feature Creation**

Rain volume lag 5 and 10 minutes.

In [ ]:
pd_merged_df["avant_port_lag1_mm"] = pd_merged_df["avant_port_mm"].shift(1)
pd_merged_df["flagey_lag1_mm"] = pd_merged_df["flagey_mm"].shift(1)

pd_merged_df["avant_port_lag2_mm"] = pd_merged_df["avant_port_mm"].shift(2)
pd_merged_df["flagey_lag2_mm"] = pd_merged_df["flagey_mm"].shift(2)

pd_merged_df[["avant_port_lag1_mm", "avant_port_lag2_mm", "flagey_lag1_mm", "flagey_lag1_mm"]].head()

Sum of different intervals.

In [ ]:
pd_merged_df["avant_port_sum_1h_mm"] = pd_merged_df["flagey_mm"].rolling(12).sum()
pd_merged_df["flagey_sum_1h_mm"] = pd_merged_df["flagey_mm"].rolling(12).sum()

pd_merged_df["avant_port_sum_30min_mm"] = pd_merged_df["flagey_mm"].rolling(6).sum()
pd_merged_df["flagey_sum_30min_mm"] = pd_merged_df["flagey_mm"].rolling(6).sum()

pd_merged_df["avant_port_sum_15min_mm"] = pd_merged_df["flagey_mm"].rolling(3).sum()
pd_merged_df["flagey_sum_15min_mm"] = pd_merged_df["flagey_mm"].rolling(3).sum()

pd_merged_df[["avant_port_sum_1h_mm", "avant_port_sum_30min_mm", "avant_port_sum_15min_mm",
             "flagey_sum_1h_mm", "flagey_sum_30min_mm", "flagey_sum_15min_mm"]].tail()

Rain intensity (per hour).

In [ ]:
pd_merged_df["avant_port_intensity_15min_mm"] = pd_merged_df["avant_port_mm"].rolling(3).sum() / (15/60)
pd_merged_df["flagey_intensity_15min_mm"] = pd_merged_df["flagey_mm"].rolling(3).sum() / (15/60)

pd_merged_df["avant_port_intensity_1h_mm"] = pd_merged_df["avant_port_mm"].rolling(3).sum() / 1
pd_merged_df["flagey_intensity_1h_mm"] = pd_merged_df["flagey_mm"].rolling(3).sum() / 1

pd_merged_df["avant_port_intensity_1d_mm"] = pd_merged_df["avant_port_mm"].rolling(3).sum() / 24
pd_merged_df["flagey_intensity_1d_mm"] = pd_merged_df["flagey_mm"].rolling(3).sum() / 24

pd_merged_df[["avant_port_intensity_15min_mm", "avant_port_intensity_1h_mm", "avant_port_intensity_1d_mm",
             "flagey_intensity_15min_mm", "flagey_intensity_1h_mm", "flagey_intensity_1d_mm"]].tail()

Seasons.

In [ ]:
months = pd_merged_df['datetime'].dt.month

bins = [0, 3, 6, 9, 12]
labels = [0, 1, 2, 3]

pd_merged_df['season'] = pd.cut(months, bins=bins, labels=labels)

pd_merged_df[["datetime", "season"]].sample(4)

In [ ]:
pd_merged_df = pd_merged_df.sort_values("datetime").reset_index(drop=True)
pd_merged_df = pd_merged_df.set_index('datetime')
pd_merged_df.head(1)

##**Sewage data preprocessing**

In [ ]:
sd_2022_df = pd.read_csv(U24_2022_path)
sd_2023_df = pd.read_csv(U24_2023_path)
sd_2024_df = pd.read_csv(U24_2024_path)

In [ ]:
sewage_complete = pd.concat([sd_2022_df, sd_2023_df, sd_2024_df], ignore_index=True)
sewage_complete = sewage_complete.rename(columns={"Date":"datetime", "Value":"waterlevel"})
sewage_complete.head()

In [ ]:
sewage_complete['datetime'] = pd.to_datetime(sewage_complete['datetime'], errors='coerce')
sewage_complete = sewage_complete.sort_values("datetime").reset_index(drop=True)


**Exploratory Data Analysis**

In [ ]:
print("head of dataset: ")
print(sewage_complete.head())

In [ ]:
print("tail of dataset: ")
print(sewage_complete.tail())

In [ ]:
print("Null values: ")
print(sewage_complete.isna().any())

We have no missing values in the 'unclean' dataset, so we don't have to interpolate or remove any.

In [ ]:
print("duplicated rows?")
print(sewage_complete.duplicated().any())

Good news, no duplicated rows either.

In [ ]:
print("summary statistics")
print(sewage_complete.describe())


**NOTE** There are negative values in the waterlevel, which are unrealistic. We have to take care of these. Let's first check how many we have and the distribution of negative values. 

In [ ]:
neg_df = sewage_complete[sewage_complete['waterlevel'] < 0]

print("amount of negative values: ", len(neg_df))

In [ ]:
neg_df['dateOnly'] = pd.to_datetime(neg_df['datetime']).dt.date
print(neg_df.groupby('dateOnly').size())


In 2022, we have a 5 day gap of negative values, 04-06-2022 to 05-06-2022. Since the gap is multiple days, it indicates a fault with the sensor. This data is unusable and should NOT be trained on. However, we will set the negative values to NaN for now, in order not to mess with the historical context and remove (or flag) them later after creating our input features (especially lagged water level).

For 2023, since there are only 3 negative values. These can potentially be interpolated. Let's check these further.

In [ ]:
print(neg_df[pd.to_datetime(neg_df['datetime']).dt.date == pd.Timestamp('2023-12-19').date()])

In [ ]:
print(sewage_complete[(pd.to_datetime(sewage_complete['datetime']) > pd.to_datetime('2023-12-19 11:45')) 
    & (pd.to_datetime(sewage_complete['datetime']) < pd.to_datetime('2023-12-19 12:00'))])

In [ ]:
sewage_complete.isna().any()

Above you can see that the three negative waterlevel values are situated in subsequent timesteps (11:50-11:52). Before this, the waterlevel was 1208.934 and after, it amounted to 1216.354. Given that there is not a big change between them and also before and after the changes to waterlevel are minimal, we can use linear interpolation for these. 


In [ ]:
# create a mask for the day and negative values
negative_2023_mask = ((sewage_complete['datetime'].dt.date == pd.Timestamp('2023-12-19').date())
    & (sewage_complete['waterlevel'] < 0))

# set to NaN
sewage_complete.loc[negative_2023_mask, 'waterlevel'] = np.nan

# check how many NaNs (should be 3)
print("NaNs before interpolation: ", sewage_complete['waterlevel'].isna().sum())

# interpolate
sewage_complete['waterlevel'] = sewage_complete['waterlevel'].interpolate(method='linear')

# check how many NaNs after (should be 0)
print("NaNs after interpolation: ", sewage_complete['waterlevel'].isna().sum())


In [ ]:
print(sewage_complete[(pd.to_datetime(sewage_complete['datetime']) > pd.to_datetime('2023-12-19 11:45')) 
    & (pd.to_datetime(sewage_complete['datetime']) < pd.to_datetime('2023-12-19 12:00'))])

As we can see above, the negative values are interpolated and the timeseries looks reasonable now.

**Clean up faulty 5 day period**

In [ ]:
sewage_complete.loc[sewage_complete['waterlevel'] < 0, 'waterlevel'] = np.nan

Now, let's have a look at how the data looks (without negative waterlevels). 

In [ ]:
# Create the plot
plt.figure(figsize=(12, 6))
plt.plot(sewage_complete['datetime'], sewage_complete['waterlevel'], linewidth=1)

# Add labels and formatting
plt.xlabel('Date')
plt.ylabel('Water Level (mm)')
plt.title('Sewer Water Levels Over Time')
plt.grid(True, alpha=0.3)

# Rotate x-axis labels for readability
plt.xticks(rotation=45)
plt.tight_layout()

# Show or save
plt.show()

From context, we know that when the level reaches 3m, an overflow event occurs. In the three years, there seems to have been quite a lot of overflow events. The waterlevel even exceeds 3.5m.

**Water level changes**

Apart from negative values, we will also check for other abnormal data, such as sudden spikes in water level that are unrealistic. For that we need the delta, aka the change of waterlevel per min.

In [ ]:
sewage_complete['delta'] = sewage_complete['waterlevel'].diff()
# do not calculate delta if waterlevel is NaN
sewage_complete.loc[sewage_complete['waterlevel'].isna(), 'delta'] = np.nan

sewage_complete.head()


In [ ]:
sewage_complete.describe()

In [ ]:
pd_merged_df = pd_merged_df.reset_index(drop=False)

In [ ]:
def plot_waterlevel_changes_and_rain(sewage, rain):
    # Create figure with 2 rows, 1 column (subplots on top of each other)
    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(12, 6))
    
    ax1.scatter(sewage['datetime'], sewage['delta'], s=1, alpha=0.5)
    ax1.set_xlabel('Date')
    ax1.set_ylabel('Change per minute (mm)')
    ax1.set_title('Water Level Changes Over Time')
    ax1.legend()
    
    ax2.plot(sewage['datetime'], sewage["waterlevel"])
    ax2.set_ylabel("Waterlevel (mm)")
    ax2.set_title("Waterlevels")
    ax2.legend()
    ax2.grid(True, alpha=0.3)
        
    ax3.plot(rain['datetime'], rain["avant_port_mm"], "steelblue", linewidth=0.8, label="Avant Port")
    ax3.plot(rain['datetime'], rain["flagey_mm"], "crimson", linewidth=0.8, label="Flagey")
    
    ax3.set_ylabel("Rainfall (mm)")
    ax3.set_title("Rainfall Comparison: Avant Port vs Flagey")
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    fig.subplots_adjust(hspace=0.5) 
    
    return fig

In [ ]:
plot_waterlevel_changes_and_rain(sewage_complete, pd_merged_df).show()

In [ ]:
# Create figure with 2 rows, 1 column (subplots on top of each other)
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 5))

ax1.scatter(sewage_complete['datetime'], sewage_complete['delta'], s=1, alpha=0.5)
ax1.set_xlabel('Date')
ax1.set_ylabel('Change per minute (mm)')
ax1.set_title('Water Level Changes Over Time')
ax1.legend()

ax2.plot(pd_merged_df["avant_port_mm"], "steelblue", linewidth=0.8, label="Avant Port")
ax2.plot(pd_merged_df["flagey_mm"], "crimson", linewidth=0.8, label="Flagey")

ax2.set_ylabel("Rainfall (mm)")
ax2.set_title("Rainfall Comparison: Avant Port vs Flagey")
ax2.legend()
ax2.grid(True, alpha=0.3)

fig.subplots_adjust(hspace=0.5) 

fig.show()

From the figure above, we can see that the moments where the delta has very high values of change often correspond to spikes in rainfall from either one or both of the sensors. This could generally already indicate that the sudden big changes in waterlevel are not faulty.

However, let's still have a look at the timeseries where changes are very high (above 1.5m).

In [ ]:
print(f"99th percentile: {sewage_complete['delta'].quantile(0.99):.1f} mm/min")
print(f"99.9th percentile: {sewage_complete['delta'].quantile(0.999):.1f} mm/min")
print(f"Max delta: {sewage_complete['delta'].max():.1f} mm/min")

In [ ]:
def show_timeseries_mins_around(timestamp, minutes):
    time = pd.to_datetime(timestamp)
    start = time - pd.Timedelta(minutes=minutes)
    end = time + pd.Timedelta(minutes=minutes)
    print(sewage_complete[(sewage_complete['datetime'] > pd.to_datetime(start)) &
                             (sewage_complete['datetime'] < pd.to_datetime(end))])

In [ ]:
def get_timeseries_mins_around(data, timestamp, minutes):
    time = pd.to_datetime(timestamp)
    start = time - pd.Timedelta(minutes=minutes)
    end = time + pd.Timedelta(minutes=minutes)
    return (data[(data['datetime'] > pd.to_datetime(start)) &
                             (data['datetime'] < pd.to_datetime(end))])

In [ ]:
# check >1.5m/min
print(">1.5m/min or <-1.5m/min")
print(sewage_complete[(sewage_complete['delta'] > 1500) | (sewage_complete['delta'] < -1500)])


In [ ]:
start_2022_03_03 = '2022-03-03 08:00'
end_2022_03_03 = '2022-03-03 08:30'
march_3_2022_sewage = get_timeseries_mins_around(sewage_complete, '2022-03-03 08:15:33', 30)
march_3_2022_rain = get_timeseries_mins_around(pd_merged_df, '2022-03-03 08:15:33', 30)


plot_waterlevel_changes_and_rain(march_3_2022_sewage, march_3_2022_rain).show()

The crazy spikes of >1.5m per minute are all on one day. Generally, this day has a lot of variation. We see that the big spikes are bringing level to exceed 3m and then to fall below 3m again. It could be realistic, considering the overflow event. 

**TO CHECK** does an overflow event mean that waterlevel falls super quickly? a meter a minute? 

TODO: check also other abnormal days

In [ ]:
# check >1.5m/min
print(">1m/min or <-1m/min")
print(sewage_complete[(sewage_complete['delta'] > 1000) | (sewage_complete['delta'] < -1000)])

In [ ]:
show_timeseries_mins_around('2022-05-19 11:36:33', 10)

In [ ]:
may_19_2022_sewage = get_timeseries_mins_around(sewage_complete, '2022-05-19 11:36:33', 1260)
may_19_2022_rain = get_timeseries_mins_around(pd_merged_df, '2022-05-19 11:36:33', 1260)
# have to unset the index of the rain one 

plot_waterlevel_changes_and_rain(may_19_2022_sewage, may_19_2022_rain).show()

Here, the fall of 1m centers happens when the waterlevel was above 3.7m, which could be explained by an overflow event, where water quickly flows out of the sewer. 

In [ ]:
show_timeseries_mins_around('2023-02-23 12:10:22', 10)


In [ ]:
feb_2_2023_sewage = get_timeseries_mins_around(sewage_complete, '2023-02-23 12:10:22', 60)
feb_2_2023_rain = get_timeseries_mins_around(pd_merged_df, '2023-02-23 12:10:22', 60)
# have to unset the index of the rain one 

plot_waterlevel_changes_and_rain(feb_2_2023_sewage, feb_2_2023_rain).show()

**NOTE** here it does NOT center around an overflow event. Can waterlevel fall 1m in a minute without an overflow event? 

In [ ]:
show_timeseries_mins_around('2023-06-20 17:58:33', 10)


In [ ]:
show_timeseries_mins_around('2024-02-19 08:16:33', 10)
show_timeseries_mins_around('2024-02-19 08:21:33', 10)

In [ ]:

show_timeseries_mins_around('2024-07-09 20:35:33', 10)

In [ ]:
print("number of NaNs: ", sewage_complete['waterlevel'].isna().sum())
print("% of NaNs: ", sewage_complete['waterlevel'].isna().sum()/len(sewage_complete) * 100)

Now we see that we have 0.43% of the data that is NaN. We have to see what to do with that data.

## Downsampling
We need to downsample to 5 mins in order to match the other rainfall dataset. We can use the mean, max or median here (median is more robust to outliers). Alternatively, we could keep multiple columns and see which one works better in our models.

In [ ]:
sewage_5min_mean = sewage_complete.resample('5min', on='datetime').mean().reset_index()
sewage_5min_max = sewage_complete.resample('5min', on='datetime').max().reset_index()


print(sewage_5min_mean.head())
print(sewage_5min_max.head())


In [ ]:
plt.figure(figsize=(14, 5))
plt.scatter(sewage_5min_max['datetime'], sewage_5min_max['delta'], s=1, alpha=0.5)
plt.xlabel('Date')
plt.ylabel('Change per minute (mm)')
plt.title('Water Level Changes WITH MAX DOWNSAMPLING Over Time')
plt.legend()
plt.show()

In [ ]:
# Create the plot
plt.figure(figsize=(12, 6))
plt.plot(sewage_5min_max['datetime'], sewage_5min_max['waterlevel'], linewidth=1)

# Add labels and formatting
plt.xlabel('Date')
plt.ylabel('Water Level (mm)')
plt.title('Sewer Water Levels 5min MAX Over Time')
plt.grid(True, alpha=0.3)

# Rotate x-axis labels for readability
plt.xticks(rotation=45)
plt.tight_layout()

# Show or save
plt.show()

In [ ]:
plt.figure(figsize=(14, 5))
plt.scatter(sewage_5min_mean['datetime'], sewage_5min_mean['delta'], s=1, alpha=0.5)
plt.xlabel('Date')
plt.ylabel('Change per minute (mm)')
plt.title('Water Level Changes 5min WITH MEAN DOWNSAMPLING Over Time')
plt.legend()
plt.show()

In [ ]:
# Create the plot
plt.figure(figsize=(12, 6))
plt.plot(sewage_5min_mean['datetime'], sewage_5min_mean['waterlevel'], linewidth=1)

# Add labels and formatting
plt.xlabel('Date')
plt.ylabel('Water Level (mm)')
plt.title('Sewer Water Levels 5min MEAN Over Time')
plt.grid(True, alpha=0.3)

# Rotate x-axis labels for readability
plt.xticks(rotation=45)
plt.tight_layout()

# Show or save
plt.show()

For the sake of the models, for now we keep the "mean" downsampling. Later, we can experiment also with the max and see how it affects model performance.

#Feature Engineering

We want to make the following features:
* lagged waterlevel (t-1, t-2, etc.)
* water level rate of change (already done)
* distance to threshold (3000-waterlevel at time t)


In [ ]:
sewage_5min_mean['waterlevel_lag_1'] = sewage_5min_mean['waterlevel'].shift(1)
sewage_5min_mean['waterlevel_lag_2'] = sewage_5min_mean['waterlevel'].shift(2)
sewage_5min_mean['waterlevel_lag_3'] = sewage_5min_mean['waterlevel'].shift(3)
sewage_5min_mean['threshold_distance'] = 3000 - sewage_5min_mean['waterlevel']
sewage_5min_mean.head()

In [ ]:
sewage_5min_mean = sewage_5min_mean.set_index('datetime')


In [ ]:
print(sewage_5min_mean.head())

In [ ]:
sewage_5min_mean.isna().sum()

#Merging Datasets

In [ ]:
pd_merged_df = pd_merged_df.set_index('datetime')
sewage_rainfall_df = pd_merged_df.join(sewage_5min_mean, how='inner')

In [ ]:
sewage_rainfall_df.head()

Let's remove the rows with NaN

In [ ]:
sewage_rainfall_df.describe()

Here we set the target (the data that needs to be predicted) - 60min/1hour (12*5) prediction.

In [ ]:
sewage_rainfall_df['target'] = sewage_rainfall_df['waterlevel'].shift(-12)

print(sewage_rainfall_df['target'].head())
print(sewage_rainfall_df['waterlevel'].head())

We remove the rows for which we cannot have a target.

In [ ]:
print(sewage_rainfall_df.isna().sum()['target'])
sewage_rainfall_df = sewage_rainfall_df.dropna()
print(sewage_rainfall_df.isna().sum()['target'])

We split the dataset into training/validation and test sets while respecting the time order, since this is time series data and shuffling would break the temporal structure. The most recent 20% of the data is kept as the test set, and the remaining 80% is used for training and validation. Furthermore, it will be split into a train and validation set that will only be useful later for the other two models. For the sweep config, it is not needed since the validation process is taking place in the train_val set. 

In [ ]:
df_train_val, df_test = train_test_split(sewage_rainfall_df, test_size=0.2, shuffle=False)

In [ ]:
X_train_val = df_train_val.drop(columns=['target'])
y_train_val = df_train_val['target']

X_test = df_test.drop(columns=['target'])
y_test = df_test['target']

In [ ]:
# make a distinguished train and val set for the other models
df_train, df_val = train_test_split(df_train_val, test_size=0.25, shuffle=False)
X_train = df_train.drop(columns=['target'])
y_train = df_train['target']
X_val = df_val.drop(columns=['target'])
y_val = df_val['target']

In [ ]:
corr_matrix = X_train_val.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f")

plt.title("Feature Correlation Matrix")
plt.show()

**Feature importance**

Select top 10 features using mutual information for regression as well as RandomForestRegressor.

In [ ]:
selector = SelectKBest(score_func=mutual_info_regression, k=10)
selector.fit(X_train_val, y_train_val)

selected_features = X_train_val.columns[selector.get_support()]
print('Selected features (SelectKBest): \n', selected_features.tolist())

In [ ]:
# Create and fit the model
rf = RandomForestRegressor(random_state=random_state)
rf.fit(X_train, y_train)

# Retrieve feature importances thourgh the feature_importances_ attribute
importances = rf.feature_importances_

# Print the top 10 features
indices = np.argsort(importances)[::-1]
print('Top 10 features by importance:')
for f in range(10):
    print(f"{f + 1}. {X_train.columns[indices[f]]} ({importances[indices[f]]:.4f})")

# Plot feature importances
plt.figure(figsize=(8,4))
plt.title('Feature Importances (Random Forest)')
sns.barplot(x=importances[indices[:10]], y=X_train.columns[indices[:10]])
plt.show()

**NOTE** We run into a problem here: the fact that threshold distance has a huge importance of 0.6 means that we are likely creating a bias here and letting the model take the shortcut of threshold distance instead of learning the relationships with the other variables. We will remove threshold distance and redo.

In [ ]:
# TODO here we want to remove threshold distance
X_train = X_train.drop(columns=['threshold_distance'])
X_val = X_val.drop(columns=['threshold_distance'])
X_test = X_test.drop(columns=['threshold_distance'])
X_train_val = X_train_val.drop(columns=['threshold_distance'])

Let's see again which ones are now the best after excluding threshold_distance...

In [ ]:
# Create and fit the model
rf = RandomForestRegressor(random_state=random_state)
rf.fit(X_train, y_train)

# Retrieve feature importances thourgh the feature_importances_ attribute
importances = rf.feature_importances_

# Print the top 10 features
indices = np.argsort(importances)[::-1]
print('Top 10 features by importance:')
for f in range(10):
    print(f"{f + 1}. {X_train.columns[indices[f]]} ({importances[indices[f]]:.4f})")

# Plot feature importances
plt.figure(figsize=(8,4))
plt.title('Feature Importances (Random Forest)')
sns.barplot(x=importances[indices[:10]], y=X_train.columns[indices[:10]])
plt.show()

In [ ]:
def compute_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return {
        "mse": mse,
        "rmse": np.sqrt(mse),
        "mae": mae,
        "r2": r2
    }

In [ ]:
def log_split_metrics(run, prefix, y_true, y_pred, step=None):
    metrics = compute_metrics(y_true, y_pred)
    metrics = {f"{prefix}/{k}": v for k, v in metrics.items()}
    run.log(metrics, step=step)

In [ ]:
os.environ["WANDB_SILENT"] = "true"

In [ ]:
wandb.login("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")

This config defines a hyperparameter sweep to optimize a model by maximizing cv/accuracy_mean. It searches over different network sizes, learning rates, batch sizes, activation functions, regularization strength, and solvers using Bayesian optimization, while keeping some training settings fixed (early stopping, validation split, and iteration limits).

In [ ]:
sweep_config = {
    "method": "bayes",  # or "random"
    "metric": {
        "name": "cv/accuracy_mean",  # must match what you log
        "goal": "maximize"
    },
    "parameters": {
        "hidden_layer_sizes": {
            "values": [
                (16,),
                (32,),
                (64,),
                (128,),
                (32, 16),
                (64, 32),
                (128, 64),
                (64, 32, 16)
            ]
        },

        "learning_rate_init": {
            "distribution": "log_uniform_values",
            "min": 1e-5,
            "max": 5e-2
        },

        "batch_size": {
            "values": [16, 32, 64, 128, 256]
        },

        "activation": {
            "values": ["relu", "tanh", "logistic"]
        },

        "alpha": {
            "distribution": "log_uniform_values",
            "min": 1e-6,
            "max": 1e-2
        },

        "solver": {
            "values": ["adam", "sgd"]
        },

        # Fixed parameters
        "early_stopping": {"value": True},
        "validation_fraction": {"value": 0.1},
        "max_iter": {"value": 80},
        "n_iter_no_change": {"value": 10}
    }
}

In [ ]:
pprint.pprint(sweep_config)

In [ ]:
sweep_id = wandb.sweep(sweep_config, project="ann-cv")

In [ ]:
entity = "sinakhaji5-vrije-universiteit-brussel"
project = "ann-cv"
sweep_id = sweep_id

In [ ]:
def sweep_train_cv(project, X_train_val, y_train_val, random_state, n_splits=5):
    with wandb.init(project=project) as run:
        raw_cfg = dict(run.config)

        # Filter valid params
        valid_params = MLPRegressor().get_params().keys()
        cfg = {k: v for k, v in raw_cfg.items() if k in valid_params}

        model = Pipeline([
            ("scaler", StandardScaler()),
            ("ann", MLPRegressor(**cfg)),
        ])

        cv = KFold(n_splits=n_splits, shuffle=False)
        scores = cross_val_score(
            model, X_train_val, y_train_val,
            cv=cv, scoring="r2", n_jobs=-1
        )

        run.log({
            "cv/accuracy_mean": float(scores.mean()),
            "cv/accuracy_std": float(scores.std()),
        })

In [ ]:
wandb.agent(
    sweep_id,
    function=lambda: sweep_train_cv(
        project = project,
        X_train_val=X_train_val,
        y_train_val=y_train_val,
        random_state=random_state
    ),
    count=1
)

In [ ]:
def retrain_best_and_test(entity, project, sweep_id, X_train_val, y_train_val, X_test, y_test):
    api = wandb.Api()
    sweep = api.sweep(f"{entity}/{project}/{sweep_id}")

    best = sweep.best_run(order="cv/accuracy_mean").load_full_data()

    raw_cfg = dict(best["rawconfig"])

    valid_params = MLPRegressor().get_params().keys()
    cfg = {k: v for k, v in raw_cfg.items() if k in valid_params}

    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("ann", MLPRegressor(**cfg))
    ])

    pipe.fit(X_train_val, y_train_val)

    y_pred = pipe.predict(X_test)

    test_acc = compute_metrics(y_test, y_pred)

    print("Best run:", best["displayName"], "| test R2:", test_acc['r2'])

    return pipe, cfg, test_acc

In [ ]:
pipe, best_hparams, test_acc = retrain_best_and_test(
    entity=entity,
    project=project,
    sweep_id=sweep_id,
    X_train_val=X_train_val,
    y_train_val=y_train_val,
    X_test=X_test,
    y_test=y_test,
)

print("Best hyperparameters found by the sweep:")
pprint.pprint(best_hparams)

In [ ]:
test_acc

In [ ]:
api = wandb.Api()
sweep = api.sweep(f"{entity}/{project}/{sweep_id}")
sweep.best_run(order="val/accuracy").name

**Other models**

Here we will train a random forest and a linear regression on the same data.


In [ ]:
# drop the columns that are not in the 10 best
best_columns=X_train.columns[indices[:10]]
X_train = df_train[best_columns]
X_val = df_val[best_columns]
X_test_best = df_test[best_columns]

print(X_train.columns)

##Train RandomForestRegressor
Here we will train on the training set, but use RandomizedSearchCV with internal cross-validation.

In [ ]:

# Create a Random Forest pipeline with RandomizedSearchCV
rf_pipe = Pipeline([
    #selector has been done previously
    #scaling is not needed in RandomForests (https://kundan-reads.readthedocs.io/en/latest/aiml/machine_learning/random_forest_regressor/)
    ('regressor', RandomForestRegressor(random_state=random_state))
])
# Define parameter distribution for RandomizedSearchCV
param_dist_rf = {
    'regressor__n_estimators': randint(50, 100),
    'regressor__max_depth': [3, 5, 7, 10],
    'regressor__min_samples_split': randint(2, 11),
    'regressor__min_samples_leaf': [1, 2, 4]
}
# Perform Randomized Search with cross-validation (cv)
rf_random_search = RandomizedSearchCV(rf_pipe,
                                      param_distributions=param_dist_rf,
                                      n_iter=20,
                                      cv=3,
                                      scoring='r2',
                                      random_state=random_state,
                                      n_jobs=4)
rf_random_search.fit(X_train, y_train)

# Print best parameters and best cross-validation score
print('Best parameters (RF):', rf_random_search.best_params_)
print('Best cross-val accuracy (RF):', round(rf_random_search.best_score_, 4))

##Train a linear model

In [ ]:

# Create a Linear Regression pipeline
lr_pipe = Pipeline([
    ('scaler', StandardScaler()), #maybe we have to take a different one bc of many outliers?
    ('regressor', LinearRegression())
])

lr_pipe.fit(X_train, y_train)

scores = cross_val_score(lr_pipe, X_train, y_train, cv=3, scoring='r2')
print('Cross-val R2 (LR):', scores)

In [ ]:
# select the best estimator from the random search for RF
best_rf = rf_random_search.best_estimator_

# predict on validation set
y_val_pred_rf = best_rf.predict(X_val)
y_val_pred_lr = lr_pipe.predict(X_val)

# check how the models score on the validation test
print('\nRandom Forest:\n')
print('Validation MSE (RF):', round(mean_squared_error(y_val, y_val_pred_rf),4))
print('Validation RMSE (RF):', round(np.sqrt(mean_squared_error(y_val, y_val_pred_rf)),4))
print('Validation MAE (RF):', round(mean_absolute_error(y_val, y_val_pred_rf),4))
print('Validation R2 (RF):', round(r2_score(y_val, y_val_pred_rf),4))
print('\nLinear Regression:\n')
print('Validation MSE (LR):', round(mean_squared_error(y_val, y_val_pred_lr),4))
print('Validation RMSE (LR):', round(np.sqrt(mean_squared_error(y_val, y_val_pred_lr)),4))
print('Validation MAE (LR):', round(mean_absolute_error(y_val, y_val_pred_lr),4))
print('Validation R2 (LR):', round(r2_score(y_val, y_val_pred_lr),4))


Looking at the validation scores, the random forest scores best, let's evaluate that one on the test set.

In [ ]:
y_test_pred_rf = best_rf.predict(X_test_best)

# check how the model scores on the test test
print('\nRandom Forest:\n')
print('Test MSE (RF):', round(mean_squared_error(y_test, y_test_pred_rf),4))
print('Test RMSE (RF):', round(np.sqrt(mean_squared_error(y_test, y_test_pred_rf)),4))
print('Test MAE (RF):', round(mean_absolute_error(y_test, y_test_pred_rf),4))
print('Test R2 (RF):', round(r2_score(y_test, y_test_pred_rf),4))


TODO:
- test on test set for random forest and linear reg
- classify based on the best model
- t+15 model

In [ ]:
TODO: i can still plot the residuals to see when the errors are high 

# References

Shakil, A., Khalighi, M. A., Pudlo, P., Leclerc, C., Laplace, D., Hamon, F., & Boudonne, A. (2023). Outlier detection in non-stationary time series applied to sewer network monitoring. Internet of Things, 21, 100654.
